## 🔗 LangChain Lab
This notebook demonstrates the basics of using LangChain for agentic workflows in e-commerce applications.

In [ ]:
pip uninstall langchain-openai chromadb

Found existing installation: langchain-openai 0.3.14
Uninstalling langchain-openai-0.3.14:
  Would remove:
    /opt/anaconda3/envs/whisper_env/lib/python3.10/site-packages/langchain_openai-0.3.14.dist-info/*
    /opt/anaconda3/envs/whisper_env/lib/python3.10/site-packages/langchain_openai/*
Proceed (Y/n)? 

In [12]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = TextLoader('sample.txt')
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = True)
chunks = text_splitter.split_documents(documents)





In [16]:
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

vector_store= Chroma.from_documents(chunks, HuggingFaceEmbeddings())
retriever = vector_store.as_retriever()

/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_23036/1632173495.py:4: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vector_store= Chroma.from_documents(chunks, HuggingFaceEmbeddings())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

# Load a small open-source LLM
hf_pipeline = pipeline("text-generation", model="distilgpt2", max_length=100)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Create the QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)


Device set to use mps:0
/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_23036/1717825689.py:7: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [26]:
query = "who are you"
result = qa_chain({"query": query})
print(result['result'])
print("sources:", [doc.metadata for doc in result["source_documents"]])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

You name is Bunny

Question: who are you
Helpful Answer: 1.2.2, I am the original person, and it's not the first person to ask me questions as I was the first person
Thank you for the time you have provided me with this question and
sources: [{'source': 'sample.txt'}]


In [50]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)
result = conversational_chain({"question": "Can you explain the main topic further?"})
print(result["answer"])

Device set to use mps:0


InternalError: Error getting collection: Database error: error returned from database: (code: 1) no such table: collections

In [8]:
ollama pull llama3

SyntaxError: invalid syntax (2452604116.py, line 1)

In [10]:
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama

# Initialize open-source LLM
llm = Ollama(model="llama3")

# Define prompt template
template = PromptTemplate(
    input_variables=["text", "word_count"],
    template="Summarize the following text in {word_count} words: {text}"
)

# Create chain
chain = template | llm

# Run the chain
text = "LangChain is a framework for building applications with large language models. It supports retrieval-augmented generation, agents, and memory."
result = chain.invoke({"text": text, "word_count": 10})
print(result)

Framework for building apps using large language models features.


In [18]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA

# Load and split document
loader = TextLoader("sample_document.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Create vector store with open-source embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Initialize LLM
llm = Ollama(model="llama3")

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# Query the system
query = "What is my name ?"
result = qa_chain({"query": query})
print("Answer:", result["result"])
print("Sources:", [doc.page_content[:100] for doc in result["source7 source_documents"])

SyntaxError: closing parenthesis ')' does not match opening parenthesis '[' (695575494.py, line 34)

In [26]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA

# Load and split document
loader = TextLoader("sample.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Create vector store with open-source embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Initialize LLM
llm = Ollama(model="llama3")

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# Query the system
query = "What is my name ?"
result = qa_chain({"query": query})
print("Answer:", result["result"])
#print("Sources:", [doc.page_content[:100] for doc in result["source7 source_documents"])

/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_2097/1078015124.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_2097/1078015124.py:32: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": query})


Answer: Your name is Bunny!


In [38]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

# Load and split document
loader = TextLoader("sample.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Create vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    collection_name="my_collection",
    persist_directory="./chroma_db"  # or set to None for in-memory only
)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Initialize LLM and memory
llm = Ollama(model="llama3")
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Create conversational RAG chain
conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

# Ask questions
questions = [
    "What is LangChain?",
    "How does it support agents?"
]
for question in questions:
    result = conversational_chain({"question": question})
    print(f"Q: {question}\nA: {result['answer']}\n")

/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_2097/512391460.py:27: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


Q: What is LangChain?
A: According to the context, LangChain is a framework for building applications with Large Language Models (LLMs).

Q: How does it support agents?
A: I'm Bunny! According to the context, LangChain supports agents, which means that it provides a way to build and integrate AI-powered agents into your application using Large Language Models (LLMs).



In [42]:
from langchain.tools import Tool
from langchain.agents import initialize_agent
from langchain_community.llms import Ollama

# Define a math tool
def calculate(expression: str) -> str:
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

math_tool = Tool(
    name="Calculator",
    func=calculate,
    description="Evaluates mathematical expressions."
)

# Initialize LLM and agent
llm = Ollama(model="llama3")
tools = [math_tool]
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type="zero-shot-react-description",
    verbose=True
)

# Run a query
result = agent.run("What is 5 * 3 + 2?")
print(result)

/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_2097/2272407145.py:21: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(
/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_2097/2272407145.py:29: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = agent.run("Wha



> Entering new AgentExecutor chain...
Let's break it down step by step.

Thought: To evaluate this expression, I'll use the Calculator tool. I need to prioritize the operations based on their order of operations (PEMDAS).

Action: Calculator
Action Input: "5 * 3 + 2"
Observation: 17
Thought:Here's my response:

Question: What is 5 * 3 + 2?
Thought: Let's break it down step by step.

Action: Calculator
Action Input: "5 * 3 + 2"
Observation: 17
Thought:I now know the final answer

Final Answer: 17

> Finished chain.
17


In [44]:
import sqlite3
from langchain.tools import Tool
from langchain.agents import initialize_agent
from langchain_community.llms import Ollama

# Define database tool
def query_employees(query: str) -> str:
    conn = sqlite3.connect("employees.db")
    cursor = conn.cursor()
    try:
        cursor.execute(query)
        results = cursor.fetchall()
        return str(results)
    except Exception as e:
        return f"Error: {str(e)}"
    finally:
        conn.close()

db_tool = Tool(
    name="EmployeeDB",
    func=query_employees,
    description="Queries employee database with SQL."
)

# Initialize LLM and agent
llm = Ollama(model="llama3")
tools = [db_tool]
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type="zero-shot-react-description",
    verbose=True
)

# Run a query
result = agent.run("List employees with role 'Developer'.")
print(result)



> Entering new AgentExecutor chain...
Let's get started!

Thought: To list employees with a specific role, we need to query the employee database. We can use the EmployeeDB function to execute a SQL query that filters employees by their role.

Action: EmployeeDB
Action Input: "SELECT * FROM employees WHERE role = 'Developer';"
Observation: Error: no such table: employees
Thought:Let's think about this...

Thought: Hmm, it looks like the employee database doesn't have a table called "employees". We need to figure out what tables are available in the database so we can modify our query accordingly.

Action: EmployeeDB
Action Input: "SHOW TABLES;"
Observation: Error: near "SHOW": syntax error
Thought:Let's think about this...

Thought: Hmm, it looks like the SQL command to show tables is not supported. Let me try something else.

Action: EmployeeDB
Action Input: "Tables;"
Observation: Error: near "Tables": syntax error
Thought:Thought: Okay, I see what's going on here...

Action: Employ

In [46]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

# Define response schema
response_schemas = [
    ResponseSchema(name="answer", description="The main answer"),
    ResponseSchema(name="confidence", description="Confidence level (0-1)")
]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# Create prompt
prompt = PromptTemplate(
    template="Answer the question and provide a confidence level.\n{format_instructions}\nQuestion: {question}",
    input_variables=["question"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()}
)

# Initialize LLM and chain
llm = Ollama(model="llama3")
chain = prompt | llm | output_parser

# Run a query
result = chain.invoke({"question": "What is LangChain?"})
print(result)

{'answer': 'LangChain is a language model that enables users to generate, modify, and combine text based on their input. It uses a combination of natural language processing (NLP) and deep learning techniques to create human-like text.', 'confidence': 0.9}


In [50]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import SequentialChain, LLMChain

# Initialize LLM
llm = Ollama(model="llama3")

# First chain: Summarize
summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize the following in 50 words: {text}"
)
summary_chain = LLMChain(llm=llm, prompt=summary_prompt, output_key="summary")

# Second chain: Extract keywords
keywords_prompt = PromptTemplate(
    input_variables=["summary"],
    template="Extract 3 keywords from this summary: {summary}"
)
keywords_chain = LLMChain(llm=llm, prompt=keywords_prompt, output_key="keywords")

# Combine chains
overall_chain = SequentialChain(
    chains=[summary_chain, keywords_chain],
    input_variables=["text"],
    output_variables=["summary", "keywords"]
)

# Run the chain
text = "LangChain is a framework for building applications with large language models. It supports retrieval-augmented generation, agents, and memory for context-aware, reasoning-based apps."
result = overall_chain({"text": text})
print("Summary:", result["summary"])
print("Keywords:", result["keywords"])

/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_2097/4164801100.py:13: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  summary_chain = LLMChain(llm=llm, prompt=summary_prompt, output_key="summary")


Summary: LangChain is an open-source framework for developing applications that utilize large language models. It enables the creation of intelligent systems through features like retrieval-augmented generation, agent interactions, and contextual memory, allowing for context-aware and reasoning-based applications.
Keywords: Here are 3 keywords extracted from the summary:

1. **Language Models**
2. **Intelligent Systems**
3. **Context-Aware**


In [52]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Initialize LLM
llm = Ollama(model="llama3")

# Prompt for sentiment classification
sentiment_prompt = PromptTemplate(
    input_variables=["text"],
    template="Classify the sentiment of this text as Positive, Negative, or Neutral: {text}"
)

# Prompt for response based on sentiment
response_prompt = PromptTemplate(
    input_variables=["text", "sentiment"],
    template="Given the text '{text}' with {sentiment} sentiment, provide a suitable response."
)

# Define a parsing function
def parse_sentiment(output: str) -> str:
    return output.strip().split(":")[-1].strip()

# Create LCEL chain
chain = (
    {"text": RunnablePassthrough()}
    | sentiment_prompt
    | llm
    | RunnableLambda(parse_sentiment)
    | {"sentiment": RunnablePassthrough(), "text": RunnablePassthrough()}
    | response_prompt
    | llm
)

# Run the chain
text = "I'm really excited about LangChain!"
result = chain.invoke(text)
print("Response:", result)

Response: Thank you for sharing your thoughts on the sentiment of this text! It's great to know that you would categorize it as Positive, and I agree with your reasoning - the phrase "really excited" does convey a strong enthusiasm and positivity towards LangChain. Your analysis is spot on! Would you like to discuss any other aspects of this text or move on to something else?


In [54]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Initialize LLM
llm = Ollama(model="llama3")

# Prompt for sentiment classification
sentiment_prompt = PromptTemplate(
    input_variables=["text"],
    template="Classify the sentiment of this text as Positive, Negative, or Neutral: {text}"
)

# Prompt for response based on sentiment
response_prompt = PromptTemplate(
    input_variables=["text", "sentiment"],
    template="Given the text '{text}' with {sentiment} sentiment, provide a suitable response."
)

# Define a parsing function
def parse_sentiment(output: str) -> str:
    return output.strip().split(":")[-1].strip()

# Create LCEL chain
chain = (
    {"text": RunnablePassthrough()}
    | sentiment_prompt
    | llm
    | RunnableLambda(parse_sentiment)
    | {"sentiment": RunnablePassthrough(), "text": RunnablePassthrough()}
    | response_prompt
    | llm
)

# Run the chain
text = "I'm really excited about LangChain!"
result = chain.invoke(text)
print("Response:", result)

Response: Agreed! The text clearly conveys a positive sentiment, with the use of the phrase "really excited" and words like "enthusiasm" and "anticipation", which strongly suggest a positive emotional tone towards LangChain.


In [58]:
from langchain_community.llms import Ollama
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from langchain.prompts.example_selector import LengthBasedExampleSelector

# Define examples
examples = [
    {"query": "How does RAG work in LangChain?", "label": "Technical"},
    {"query": "What is LangChain?", "label": "General"},
    {"query": "Explain embeddings.", "label": "Technical"},
    {"query": "Who created LangChain?", "label": "General"}
]

# Example prompt
example_prompt = PromptTemplate(
    input_variables=["query", "label"],
    template="Query: {query}\nLabel: {label}\n"
)

# Example selector
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=100
)

# Main prompt
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Classify the following query as Technical or General:",
    suffix="Query: {query}\nLabel:",
    input_variables=["query"]
)

# Initialize LLM
llm = Ollama(model="llama3")

# Create chain
chain = prompt | llm

# Run query
query = "How do agents work in LangChain?"
result = chain.invoke({"query": query})
print("Result:", result.strip())

Result: Technical


In [60]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.callbacks.base import BaseCallbackHandler
import time

# Custom callback handler
class LoggingCallback(BaseCallbackHandler):
    def on_chain_start(self, serialized, inputs, **kwargs):
        self.start_time = time.time()
        print(f"Chain started with inputs: {inputs}")

    def on_chain_end(self, outputs, **kwargs):
        elapsed = time.time() - self.start_time
        print(f"Chain completed in {elapsed:.2f}s. Output: {outputs}")

# Initialize LLM
llm = Ollama(model="llama3")

# Create prompt
prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize this text: {text}"
)

# Create chain with callbacks
chain = prompt | llm
callback = LoggingCallback()

# Run chain
text = "LangChain is a framework for building applications with LLMs."
result = chain.invoke({"text": text}, config={"callbacks": [callback]})
print("Summary:", result)

Chain started with inputs: {'text': 'LangChain is a framework for building applications with LLMs.'}
Chain started with inputs: {'text': 'LangChain is a framework for building applications with LLMs.'}
Chain completed in 0.00s. Output: text='Summarize this text: LangChain is a framework for building applications with LLMs.'
Chain completed in 5.31s. Output: Here is a summary of the text:

LangChain is a software framework designed to help developers build applications that utilize Large Language Models (LLMs).
Summary: Here is a summary of the text:

LangChain is a software framework designed to help developers build applications that utilize Large Language Models (LLMs).


In [2]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langsmith import Client

# Initialize LangSmith client
client = Client()

# Load and split document
loader = TextLoader("sample.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Create vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Initialize LLM
llm = Ollama(model="llama3")

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

from langsmith.tracing import traceable

@traceable(name="qa_chain")
def run_query():
    return qa_chain.invoke({"query": query})

result = run_query()


/opt/anaconda3/envs/whisper_env/lib/python3.10/site-packages/langsmith/client.py:271: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_5173/2249917709.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/var/folders/x0/f1df0j3s0cbg2_9p24wvmkp00000gn/T/ipykernel_5173/2249917709.py:24: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langc

ModuleNotFoundError: No module named 'langsmith.tracing'

In [16]:
pip install --upgrade langsmith


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.3.33
    Uninstalling langsmith-0.3.33:
      Successfully uninstalled langsmith-0.3.33
Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain_core.runnables import RunnableBranch, RunnablePassthrough
from langchain.prompts import PromptTemplate

# Load and split document
loader = TextLoader("sample.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Create vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Initialize LLM
llm = Ollama(model="llama3")

# RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

# Direct LLM chain
direct_prompt = PromptTemplate(
    input_variables=["query"],
    template="Answer this general question: {query}"
)
direct_chain = direct_prompt | llm

# Classifier to determine query type
classifier_prompt = PromptTemplate(
    input_variables=["query"],
    template="Is this query about LangChain's features? Answer 'Yes' or 'No': {query}"
)
classifier_chain = classifier_prompt | llm | (lambda x: x.strip() == "Yes")

# Create branch
branch = RunnableBranch(
    (classifier_chain, rag_chain),
    direct_chain
)

# Run queries
queries = [
    "What workflows does LangChain support?",
    "What is the capital of France?"
]
for query in queries:
    result = branch.invoke({"query": query})
    print(f"Query: {query}\nAnswer: {result}\n")

Query: What workflows does LangChain support?
Answer: {'query': 'What workflows does LangChain support?', 'result': 'According to the context, LangChain supports RAG (Reinforcement Learning from Auxiliary Goals), agents, and memory.'}

Query: What is the capital of France?
Answer: The capital of France is Paris.



In [16]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain.retrievers import BM25Retriever, EnsembleRetriever

# Load and split document
loader = TextLoader("sample.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Create vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Create BM25 retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 2

# Combine retrievers
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

# Initialize LLM
llm = Ollama(model="llama3")

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever,
    return_source_documents=True
)

# Run query
query = "What workflows does LangChain support?"
result = qa_chain.invoke({"query": query})
print("Answer:", result["result"])
print("Sources:", [doc.page_content[:100] for doc in result["source_documents"]])

Answer: Based on the provided context, I can confidently say that LangChain supports:

* RAG (Response-Action Generation)
* Agents
* Memory

So, the answer is: LangChain supports RAG, agents, and memory.
Sources: ['You name is Bunny. LangChain is a framework for building applications with LLMs. It supports RAG, ag']


In [20]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA

# Load and split document
loader = TextLoader("sample.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Create vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Initialize LLM
llm = Ollama(model="llama3")

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

# Batch process queries
queries = [
    {"query": "What workflows does LangChain support?"},
    {"query": "What is LangChain?"}
]
results = qa_chain.batch(queries)
for query, result in zip(queries, results):
    print(f"Query: {query['query']}\nAnswer: {result['result']}\n")

Query: What workflows does LangChain support?
Answer: According to the context, LangChain supports:

1. RAG (I'm assuming this stands for "Reinforcement-based Agent Guiding")
2. Agents
3. Memory

So, the helpful answer is that LangChain supports these three workflows!

Query: What is LangChain?
Answer: LangChain is a framework for building applications with Large Language Models (LLMs). It supports RAG, agents, and memory.



In [24]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Load and split document
loader = TextLoader("sample.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Create vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)
base_retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Initialize LLM
llm = Ollama(model="llama3")

# Create compressor
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,
    return_source_documents=True
)

# Run query
query = "What does LangChain support for retrieval?"
result = qa_chain.invoke({"query": query})
print("Answer:", result["result"])
print("Sources:", [doc.page_content for doc in result["source_documents"]])

Answer: According to the context, LangChain supports RAG (Retrieve, Aggregate, Generate).
Sources: ['>>> You name is Bunny. LangChain is a framework for building applications with LLMs. It supports RAG, agents, and memory.\n\nReturned extracted context: >>> LangChain is a framework for building applications with LLMs. It supports RAG, agents, and memory.', '*AS IS*\n\n>>> It supports RAG...\n\n(The extracted part is the only relevant information from the context that answers the question)']


In [26]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel

# Initialize LLM
llm = Ollama(model="llama3")

# Summary prompt
summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize this text in 10 words: {text}"
)

# Keywords prompt
keywords_prompt = PromptTemplate(
    input_variables=["text"],
    template="Extract 3 keywords from this text: {text}"
)

# Create parallel chain
chain = RunnableParallel(
    summary=summary_prompt | llm,
    keywords=keywords_prompt | llm
)

# Run chain
text = "LangChain supports metadata filtering and compression for efficient retrieval."
result = chain.invoke({"text": text})
print("Summary:", result["summary"])
print("Keywords:", result["keywords"])

Summary: LangChain tool optimizes data retrieval with filtering and compression.
Keywords: Here are three keywords extracted from the text:

1. Metadata
2. Filtering
3. Compression
